In [ ]:
import numpy as np
#import matplotlib
#%matplotlib notebook
import matplotlib.pyplot as plt
import pylcp.gratings as gratings
import cProfile, pstats, io

In [ ]:
zchip = 1.0
nr=3
testBeams = gratings.maskedGaussianGratingMOTBeams(delta=-1., s=1., nr=nr, thd=np.pi/4,
                                                   pol=np.array([-1/np.sqrt(2), 1j/np.sqrt(2), 0]),
                                                   reflected_pol=np.array([np.pi, 0]),
                                                   reflected_pol_basis='poincare', eta=None, eta0=None,
                                                   wb=21.0, rs=13.0, center_hole=1.5,
                                                   outer_radius=11.0, zgrating=zchip, grating_angle=0)

# Trigger numba to compile the beta code:
testBeams.beam_vector[0].beta()
testBeams.beam_vector[1].beta()

x_beta = 15
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [zchip-5, zchip-1e-9, zchip+5]

plt.figure("Laser Beams", figsize=(4, 1.5*nr))
plt.clf()
pr = cProfile.Profile()

for jj, laserBeam in enumerate(testBeams.beam_vector):
    for ii, z_test in enumerate(z_tests):
        Z = z_test*np.ones(X.shape)
        it = np.nditer([X, Y, Z, None])
        Rt=np.array([X, Y, Z])

        """pr.enable()
        for (x, y, z, beta) in it:
            beta[...] = laserBeam.beta(np.array([x, y, z]), 0.)
        pr.disable()
        
        BETA = it.operands[3]"""
        
        pr.enable()
        BETA = laserBeam.beta(Rt)
        pr.disable()
        
        plt.subplot(len(testBeams.beam_vector), len(z_tests), jj*len(z_tests)+ii+1)
        plt.imshow(BETA, origin='lower',
                   extent=(-x_beta, x_beta,
                           -x_beta, x_beta))
        plt.clim((0, 1))
        plt.set_cmap('jet')
        # Make a cross-hair:
        plt.plot([0, 0], [-x_beta, x_beta],
                 'w-', linewidth=0.25)
        plt.plot([-x_beta, x_beta], [0, 0],
                 'w-', linewidth=0.25)

In [ ]:
Z = (zchip-8)*np.ones(X.shape)

Rt=np.array([X, Y, Z])
plt.imshow(np.sum(testBeams.beta(Rt)[1::], axis=0))

In [ ]:
s = io.StringIO()
sortby = 'cumtime'
ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
ps.print_stats()
print(s.getvalue())

In [ ]:
nr=3
testBeams = gratings.infiniteGratingMOTBeams(delta=-1., s=1., nr=nr, thd=np.pi/4,
                                             pol=np.array([-1/np.sqrt(2), 1j/np.sqrt(2), 0]),
                                             reflected_pol=np.array([np.pi, 0]),
                                             reflected_pol_basis='poincare', eta=None,
                                            grating_angle=0)
x_beta = 2
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [0]

plt.figure("Infinite Beams", figsize=(4, 1.5*nr))
plt.clf()
for jj, laserBeam in enumerate(testBeams.beam_vector):
    for ii, z_test in enumerate(z_tests):
        plt.subplot(len(testBeams.beam_vector), len(z_tests), jj*len(z_tests)+ii+1)
        Rt=np.array([X, Y, z_test*np.ones(X.shape)])
        tt=np.zeros(X.shape)
        print(laserBeam.kvec())
        plt.imshow(laserBeam.beta(R=Rt,t=tt),
                   origin='lower',
                   extent=(-x_beta, x_beta,
                           -x_beta, x_beta))
        plt.clim((0, 1))
        plt.set_cmap('jet')
        # Make a cross-hair:
        plt.plot([0, 0], [-x_beta, x_beta],
                 'w-', linewidth=0.25)
        plt.plot([-x_beta, x_beta], [0, 0],
                 'w-', linewidth=0.25)